### **[_Create Streaming Tables with SQL using Auto Loader_](url)**

In this demonstration we will create a streaming table to incrementally ingest files from a volume using Auto Loader with SQL...

When you create a streaming table using the CREATE OR REFRESH STREAMING TABLE statement, the initial data refresh and population begin immediately. These operations do not consume DBSQL warehouse compute. Instead, streaming table rely on serverless DLT for both creation and refresh. A dedicated serverless DLT pipeline is automatically created and managed by the system for each streaming table.

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema, current_database() as database;

In [0]:
%sql
-- change the default database to the one created in the previous cell
USE CATALOG pysaprk_demo;

USE SCHEMA autoloader;
    
-- display the current database
SELECT current_catalog() as catalog, current_schema() as schema;

#### Create Streaming Tables for Incremental Processing

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/pysaprk_demo/autoloader/csv_files_autoloader_source/transactions_01.csv',
  format => 'csv',
  header => true,
  inferSchema => true,
  sep => ',',
  ignoreLeadingWhiteSpace => true,
  ignoreTrailingWhiteSpace => true,
  dateFormat => 'yyyy-MM-dd HH:mm:ss',
  timestampFormat => 'yyyy-MM-dd HH:mm:ss'
)

#### Create a STREAMING TABLE using Databricks SQL

In [0]:
%sql
-- create a streaming table
CREATE OR REFRESH STREAMING TABLE sql_csv_autoloader_source
SCHEDULE EVERY 1 HOUR
AS 
SELECT 
    *, 
    _metadata.file_path as file_path,
    _metadata.file_modification_time as file_modification_time,
    _metadata.file_name as file_name,
    _metadata.file_size as file_size,
    current_timestamp() as ingestion_time
FROM STREAM read_files(
  '/Volumes/pysaprk_demo/autoloader/csv_files_autoloader_source/',
  format => 'csv',
  header => true,
  inferSchema => true,
  sep => ',',
  ignoreLeadingWhiteSpace => true,
  ignoreTrailingWhiteSpace => true,
  dateFormat => 'yyyy-MM-dd HH:mm:ss',
  timestampFormat => 'yyyy-MM-dd HH:mm:ss'
);

In [0]:
%sql
-- display the streaming table
SELECT * FROM sql_csv_autoloader_source limit 10;

In [0]:
%sql
SELECT 
  file_name,
  count(*) as total_records
FROM sql_csv_autoloader_source
GROUP BY file_name limit 10;


In [0]:
%sql
DESCRIBE TABLE EXTENDED sql_csv_autoloader_source;

In [0]:
%sql
DESCRIBE HISTORY sql_csv_autoloader_source;

### Refresh streaming Table using REFRESH STREAMING TABLE table_name

In [0]:
%sql
REFRESH STREAMING TABLE sql_csv_autoloader_source;

In [0]:
%sql
SELECT file_name, ingestion_time, count(*)
FROM sql_csv_autoloader_source
GROUP BY file_name, ingestion_time
ORDER BY count(*) DESC
;

In [0]:
%sql
DESCRIBE HISTORY sql_csv_autoloader_source;

#### Drop the streaming table

In [0]:
%sql
DROP TABLE IF EXISTS sql_csv_autoloader_source;